# Universidad Central · Big Data · Sesión 13
# ¿Cuándo conviene distribuir tareas y entrenar un modelo?

**Caso principal:** Laura / Compras Claras. **Entorno:** Codespaces con el perfil S13.

[Presentación](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1) · [Datos SECOP](../Datos/tc2_corpus_2026.json)

Usaremos **SECOP real** para practicar particiones y tareas. Para enseñar **K-means**, generaremos **puntos numéricos simulados**: así se entiende el algoritmo sin transformar textos ni crear archivos Python adicionales.

**Producto:** comparación pandas–Dask, Parquet SECOP y un modelo K-means **local frente a distribuido**, ejecutados directamente en este cuaderno.

## Objetivo y recorrido

Responder cuatro preguntas: ¿qué es un worker?, ¿cómo se reparte una tarea?, ¿cómo se entrena **un único modelo clásico** entre varios workers?, ¿cuándo la distribución agrega costo en lugar de ahorrar tiempo?

**Ruta:** revisar CPU y memoria → Dask con ocho lotes SECOP → particiones y MapReduce → pandas frente a Dask → K-means numérico, local y distribuido → interpretar tamaños y tiempos.

No usamos modelos neuronales, búsqueda de textos, herramientas externas ni experimentos masivos para esta práctica.

## Cinco estaciones dentro del mismo cuaderno

1. **Codespaces:** CPU lógicas, memoria y configuración de workers.
2. **Distribución:** ocho lotes SECOP con trabajo real y espera simulada, debidamente diferenciados.
3. **MapReduce:** combinar resultados parciales de manera correcta.
4. **Datos:** pandas y Dask sobre las mismas filas; exportar Parquet.
5. **Modelo clásico:** generar puntos numéricos, entrenar K-means local y Dask-ML y decidir cuándo conviene distribuir.

Este cuaderno contiene la ejecución del modelo. No hace falta abrir Streamlit ni copiar scripts adicionales.

## 0 · Decidir antes de instalar

**Pregunta profesional:** ¿qué problema justifica pagar el costo de coordinar trabajadores?

| Trabajo | Primera opción razonable |
|---|---|
| 200 registros que caben en memoria | pandas |
| Consulta SQL sobre muchos CSV/Parquet | considerar DuckDB |
| Muchas transformaciones independientes o datos mayores que memoria | Dask puede ayudar |
| Un modelo clásico con cientos de ejemplos | scikit-learn |
| Un único modelo profundo entrenado entre máquinas | algoritmo con soporte distribuido explícito |

**OJO:** Dask no es sinónimo de «más rápido». Dos workers dentro de un Codespace **no son dos computadores**. Cada herramienta resuelve una necesidad distinta.


## 1 · Crear tu entorno propio en el navegador

**HAZ ESTO AHORA.** 1. Abre [el repositorio](https://github.com/jazaineam1/BigData2026). 2. Pulsa **Code → Codespaces → ⋯ → New with options**. 3. En configuración del contenedor selecciona **S13 · Dask, MapReduce y modelos SECOP**. **No** elijas «Airflow Lab» (predeterminado) ni «S11 multimedia». 4. Crea un Codespace y abre una **Terminal → New Terminal**.

Dentro de la terminal ejecuta:

```bash
python --version
python -m pip show pandas
python -m pip show dask
python -m pip show scikit-learn
python Laboratorios/S13_Codespaces/verificar_entorno.py
```

Si falta una biblioteca, **dentro de Codespaces**, no en tu PC: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`. Abre este cuaderno en VS Code web y selecciona el kernel Python del Codespace. No necesita Ubuntu, Docker ni VS Code instalados localmente.

**Resultados:** debe terminar con `LISTO PARA S13`; si escogiste el perfil equivocado, crea otro Codespace con la configuración S13.


### 1A · Preparar rutas (por qué no importa si abriste el cuaderno desde Cuadernos)

`Path` maneja rutas; `sys.path` indica dónde encontrar las funciones reutilizables. Conserva visible el código: puedes adaptarlo a un proyecto ajeno.


In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "Datos").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "Datos/tc2_corpus_2026.json").exists(), "Abre el repositorio en Codespaces"
sys.path.insert(0, str(RAIZ / "Laboratorios/S13_Codespaces"))
print("Raíz del curso:", RAIZ)


### 1B · Comprobar biblioteca y archivos, no sólo que pip haya respondido

Esta celda ejecuta un cálculo mínimo Dask y comprueba que existen las dos fuentes del curso.


In [ ]:
import pandas as pd
import dask.dataframe as dd
from analisis import leer_corpus, leer_etiquetados

demo = pd.DataFrame({"grupo": ["A", "A", "B"], "valor": [10, 20, 30]})
mini = dd.from_pandas(demo, npartitions=2)
print(mini.groupby("grupo")["valor"].sum().compute())
assert int(mini.groupby("grupo")["valor"].sum().compute().loc["A"]) == 30
print("Fuentes y herramientas:", len(leer_corpus()), len(leer_etiquetados()))


**Cómo se lee:** A suma 30; B suma 30. Se encuentran 200 expedientes de referencia y 1.000 procesos etiquetados.  
**Qué nos dice:** pandas y Dask están disponibles y pueden ejecutar un cálculo.  
**Qué NO permite concluir:** que haya un clúster físico o mayor rendimiento que pandas.  
**Error común:** asumir que «pip instaló» equivale a «el flujo completo funciona».


## Laboratorio central · primero un caso en que Dask puede demostrar concurrencia

**Experimento inicial del estudiante:** trabajamos con ocho lotes independientes tomados de hasta 1.000 títulos SECOP. La función calcula un conteo real y huella SHA-256; incluye una espera de 0,6 segundos **simulada y explícita**, análoga a esperar respuesta de un servicio de OCR/ASR. Dask Distributed **sí ejecuta** la función en varios procesos worker; no ejecutamos OCR ni transcripción en esta demostración.

**Qué diferencia debes observar:** el tiempo secuencial ejecuta los ocho lotes uno por uno; Dask caliente distribuye entre workers; el total añade el arranque del scheduler. El script comprueba que ambos caminos generen el mismo resultado por lote. La ganancia puede variar según VM, número de workers y carga del Codespace.

**Códigos útiles del entorno:**

```bash
nproc
lscpu | head -25
free -h
python Laboratorios/S13_Codespaces/dask_cuando_si.py --workers 2
```

Si `nproc` indica al menos cuatro CPU lógicas y la cuota permite la prueba, compara con `--workers 4`. Este laboratorio no descarga modelos ni procesa 50.000 fichas.


In [ ]:
import os
import subprocess
import sys
import json
from pathlib import Path

print("CPU lógicas del sistema:", os.cpu_count())
disponibles = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
print("CPU lógicas permitidas por afinidad:", disponibles)
from dask_cuando_si import cpus_disponibles
print("Límite operativo orientativo (afinidad + cuota cgroup):", cpus_disponibles())
print("Para inspeccionar físicamente la VM ejecuta en terminal: nproc; lscpu; free -h")
print("Una VM puede tener 1, 2 o 4 workers, cada uno con uno o varios hilos.")


In [ ]:
# Demo pedagógica pequeña: 8 tareas con Dask Distributed real y espera externa SIMULADA.
# No activa Ollama, 50.000 fichas ni ningún servicio externo.
DEMO_DASK_EN_RUN_ALL = True
WORKERS_DEMO_S13 = 2  # Cambia SOLO este número a 4 para repetir la demostración.
script_dask = RAIZ / "Laboratorios/S13_Codespaces/dask_cuando_si.py"
assert script_dask.exists(), script_dask
if DEMO_DASK_EN_RUN_ALL:
    prueba = subprocess.run(
        [sys.executable, str(script_dask), "--workers", str(WORKERS_DEMO_S13),
         "--tareas", "8", "--espera", "0.6"],
        check=True, cwd=RAIZ, capture_output=True, text=True, timeout=120,
    )
    print(prueba.stdout)
    marca = "RESULTADO_S13_DASK="
    linea = next((l[len(marca):] for l in prueba.stdout.splitlines()
                  if l.startswith(marca)), None)
    assert linea, "No se obtuvo JSON verificable de Dask"
    resumen_demo = json.loads(linea)
    assert resumen_demo["workers"] == WORKERS_DEMO_S13
    assert resumen_demo["resultados_identicos"]
    assert len(resumen_demo["workers_reales"]) == WORKERS_DEMO_S13
    print("Ganador con clúster caliente:", resumen_demo["ganador_caliente"])
    print("Ganador incluyendo arranque:", resumen_demo["ganador_total"])
else:
    print("Para activar esta demostración ligera cambia DEMO_DASK_EN_RUN_ALL=True.")


In [ ]:
# Procesamiento REAL de 1000 títulos SECOP: conteo de vocabulario por lote.
# Se usan los mismos trabajadores; no hay espera simulada, ni OCR, ni modelos.
prueba_texto = subprocess.run(
    [sys.executable, str(script_dask), "--workers", str(WORKERS_DEMO_S13),
     "--tareas", "8", "--modo", "texto"], check=True, cwd=RAIZ,
    capture_output=True, text=True, timeout=120,
)
linea = next((l.split("RESULTADO_S13_DASK=", 1)[1]
              for l in prueba_texto.stdout.splitlines()
              if l.startswith("RESULTADO_S13_DASK=")), None)
assert linea, prueba_texto.stdout[-1800:]
resumen_texto = json.loads(linea)
assert resumen_texto["resultados_identicos"]
assert sum(resumen_texto["terminos_distintos_por_lote"]) > 0
print("Títulos SECOP:", resumen_texto["filas_procesadas"])
print("Workers reales:", len(resumen_texto["workers_reales"]))
print("Segundos secuencial:", resumen_texto["segundos_secuencial"])
print("Segundos Dask caliente:", resumen_texto["segundos_dask_cluster_caliente"])
print("Segundos Dask con arranque:", resumen_texto["segundos_dask_total_incluido_arranque"])
print("Ganador con arranque:", resumen_texto["ganador_total"])


**Cómo se lee:** esta vez Dask hace una transformación auténtica: cuenta los términos de hasta 1.000 títulos SECOP, reparte lotes entre workers y verifica igualdad de diccionarios de frecuencias. No hay pausas artificiales.

**Qué nos dice:** una tarea pequeña puede perder frente a la ejecución local por el costo de iniciar y coordinar workers.

**Qué NO permite concluir:** que Dask no sirva cuando hay datos mayores, operaciones costosas o múltiples máquinas.

**Error común:** confundir ganar en una espera concurrente simulada con ganar en cualquier función de procesamiento.


**Cómo se lee:** el JSON indica número real de workers, CPU habilitadas, filas y palabras procesadas, tiempo secuencial, tiempo de arranque, tiempo Dask caliente y tiempo Dask completo.  
**Qué demuestra:** Dask ejecutó trabajo real en procesos worker y el resultado fue idéntico. La espera fue **simulada**, no es una medición real de OCR.  
**Qué no demuestra:** que Dask sea siempre más rápido en cualquier equipo o con un modelo neuronal.  
**Prueba adicional:** modifica `--workers 2` a `--workers 4` solo si la VM habilita al menos 4 CPU lógicas; observa si cambia la aceleración y el consumo de memoria.

**¿Por qué enseñamos Dask?** Porque puede distribuir funciones Python repetibles, procesar particiones de datos y coordinar tareas independientes. A partir de ahora podrás sustituir la función de demostración por una función real de extracción o clasificación por archivo.


## 2 · Sistemas de archivos: bloques, nodos y réplicas

Un sistema como **HDFS** divide archivos en bloques lógicos y almacena esos bloques en nodos. La replicación crea copias para tolerar fallos. El **NameNode** administra metadatos de ubicación; los **DataNodes** guardan bloques.

Esto NO es lo mismo que partir un DataFrame para procesarlo. La siguiente celda es una **SIMULACIÓN en Python**, no un clúster Hadoop real.

| Elemento | Función |
|---|---|
| Nodo | máquina o proceso que participa |
| Bloque | unidad de almacenamiento |
| Réplica | otra copia de un mismo bloque |
| Partición | subconjunto lógico de trabajo |
| Scheduler | coordinador de tareas |
| Worker | ejecutor de tareas |

**Decisión:** ¿qué sucede si se pierde un nodo que guarda la única copia de un bloque?


In [ ]:
ubicaciones = {
    "bloque_1": ["nodo_A", "nodo_B"],
    "bloque_2": ["nodo_B", "nodo_C"],
    "bloque_3": ["nodo_A", "nodo_C"],
}
nodo_caido = "nodo_B"

for bloque, nodos in ubicaciones.items():
    disponibles = [n for n in nodos if n != nodo_caido]
    print(bloque, "→", disponibles)
    assert disponibles, f"Bloque no recuperable: {bloque}"


**Cómo se lee:** cada bloque conserva una ubicación distinta de nodo_B.  
**Qué nos dice:** copiar bloques en distintos nodos puede proteger contra una falla simple.  
**Qué NO permite concluir:** que un Codespace sea un HDFS real ni que todas las políticas de replicación garanticen disponibilidad.  
**Error común:** confundir replicar los datos con paralelizar el cálculo.


## 3 · MapReduce: la trampa del promedio de promedios

Imagina 3 lotes de contrataciones con diferentes tamaños. En el primer lote hay 2 importes, en el segundo 5 y en el tercero 100. **Promediar las tres medias de los lotes es incorrecto** porque no representan el mismo número de observaciones.

`map` produce estadísticas parciales. `reduce` combina parciales sin perder los pesos. **Es el modelo MapReduce implementado en Python**, no un servicio de Apache Hadoop.


In [ ]:
lotes = [
    [100, 120],
    [10, 12, 11, 9, 8],
    [1] * 100,
]
parciales = [(sum(lote), len(lote)) for lote in lotes]
print("MAP (suma, cantidad):", parciales)


In [ ]:
suma_total = sum(suma for suma, cantidad in parciales)
cantidad_total = sum(cantidad for suma, cantidad in parciales)

promedio_correcto = suma_total / cantidad_total
promedio_incorrecto = sum(s / n for s, n in parciales) / len(parciales)

print("REDUCE correcto:", round(promedio_correcto, 3))
print("Promedio de promedios:", round(promedio_incorrecto, 3))
assert round(promedio_correcto, 6) == round(370/107, 6)


**Cómo se lee:** 370 unidades en 107 observaciones dan una media de aproximadamente 3,458.  
**Qué nos dice:** para un promedio distribuido necesitamos **suma y cantidad**, no sólo medias locales.  
**Qué NO permite concluir:** que ya usamos Hadoop MapReduce ni que un algoritmo cualquiera sea divisible así.  
**Error común:** `mean(mean(lote))` ignora los tamaños de las particiones.

**Transfiere:** esta estrategia sirve para métricas de ventas, llamadas o tiempos por cliente, siempre que conserves los pesos.


## 4 · Dataset preparado para demostrar particiones (sin auditoría estudiantil)

En S13 utilizaremos la **muestra SECOP versionada en el repositorio** como insumo para operaciones sobre tablas y textos. El material está preparado para el laboratorio: sus reglas de auditoría de fuente, ceros, identificadores y clasificaciones administrativas son asuntos de validación técnica docente, no actividades evaluables para el estudiante.

En clase importa **cómo Dask procesa una función sobre bloques, cómo se distribuye en varios workers, cómo se combina el resultado y cuánto tiempo cuesta hacerlo**.


In [ ]:
referencia = leer_corpus()
print("Expedientes del taller:", len(referencia))
print(referencia[["id_proceso", "entidad"]].head(3))
assert referencia["id_proceso"].is_unique


### 4A · Usar categorías para contar y agrupar (NO entrenar otro modelo)

El CSV SECOP versionado contiene `descripci_n_del_procedimiento` y `modalidad_de_contratacion`. Aquí **modalidad** se usa exclusivamente para practicar `groupby` y comparar pandas frente a Dask sobre los mismos registros.

**Objetivo observable:** que la suma de las categorías por modalidad coincida exactamente entre ambos motores, independientemente de cómo se distribuyan las particiones.

**No entrenaremos un clasificador por modalidad.** La ampliación opcional del final, si decides activarla, se refiere únicamente a categorías administrativas existentes.


In [ ]:
tabla = leer_etiquetados()
print("Dimensiones:", tabla.shape)
print(tabla["modalidad_de_contratacion"].value_counts().head(7))
print(tabla[["id_del_proceso", "descripci_n_del_procedimiento"]].head(2))
assert tabla["id_del_proceso"].is_unique


**Cómo se lee:** 1.000 filas, 59 columnas; «Contratación directa» y «Contratación régimen especial» son clases frecuentes.  
**Qué nos dice:** hay etiquetas registradas y texto para plantear una clasificación didáctica.  
**Qué NO permite concluir:** que esas frecuencias representen todo SECOP ni que el texto contenga evidencia de fraude.  
**Error común:** entrenar sobre los 200 expedientes del Taller 2 como si tuvieran esa etiqueta.


## 5 · pandas primero: la referencia de corrección (conteo de modalidades, NO clasificación)

`groupby` agrupa filas por modalidad y `size()` cuenta filas. Esta salida es la referencia contra la cual compararemos Dask.


In [ ]:
conteo_pandas = tabla.groupby("modalidad_de_contratacion").size().sort_index()
print(conteo_pandas.head(10))
print("Total:", int(conteo_pandas.sum()))
assert conteo_pandas.sum() == 1000


## 6 · Dask DataFrame: el mismo cálculo por particiones

`from_pandas` divide el DataFrame en particiones. `groupby` construye operaciones diferidas; `compute()` materializa el resultado. Una partición **no** es un servidor.

¿Cuál sería el costo de Dask en sólo mil registros? El coordinador, el grafo de tareas y la comunicación pueden tardar más que la operación real.


In [ ]:
dask_tabla = dd.from_pandas(tabla, npartitions=4)
pendiente = dask_tabla.groupby("modalidad_de_contratacion").size()

print("Particiones:", dask_tabla.npartitions)
print("Antes de compute:", type(pendiente).__name__)
conteo_dask = pendiente.compute().sort_index()
valores_pandas = {str(k): int(v) for k, v in conteo_pandas.items()}
valores_dask = {str(k): int(v) for k, v in conteo_dask.items()}
assert valores_pandas == valores_dask
print("Resultado Dask:", conteo_dask.head(), "\nCoincide: SÍ")


### 6A · Medir sin inventar «Dask siempre gana»

Una medición debe incluir el tiempo de ejecución, el número de particiones y el tamaño del conjunto. Se repite en condiciones comparables; el primer tiempo puede incluir inicialización, cachés y procesos. Comparamos por **corrección** antes de velocidad.


In [ ]:
from analisis import comparar_pandas_dask
mediciones = []
for n_particiones in (1, 2, 4):
    mediciones.append(comparar_pandas_dask(tabla, npartitions=n_particiones))

display(pd.DataFrame(mediciones)[
    ["filas", "particiones", "segundos_pandas", "segundos_dask", "coinciden"]
])


**Cómo se lee:** los segundos se miden en TU Codespace y pueden cambiar en cada ejecución.  
**Qué nos dice:** la comparación es local, sobre el mismo conjunto de 1.000 registros.  
**Qué NO permite concluir:** cuál herramienta es más rápida para millones de filas en otro sistema.  
**Error común:** confundir aumentar particiones con aumentar máquinas o comparar ejecuciones con cargas distintas.


## 10 · Guardar evidencia reutilizable con Parquet

Parquet es un formato **columnar**. El archivo siguiente mantiene procesos únicos, modalidad y descripción; no guarda por sí mismo los pesos de un modelo entrenado. La sesión 14 lo leerá con PySpark.


In [ ]:
from analisis import escribir_resultados
manifest = escribir_resultados(tabla)
print("Salida:", manifest["salida"])
print("Filas:", manifest["filas"])
print("SHA-256 fuente:", manifest["sha256_fuente"][:20], "...")

archivo = RAIZ / "Laboratorios/S13_Codespaces/salidas/secop_s13.parquet"
reabierto = pd.read_parquet(archivo)
assert reabierto["id_del_proceso"].is_unique
print("Reabierto:", reabierto.shape)


**Cómo se lee:** se exportaron 1.000 procesos únicos de la muestra y se recuperaron desde Parquet.  
**Qué nos dice:** la entrega es tabular, trazable a una fuente específica y legible.  
**Qué NO permite concluir:** que el archivo Parquet tenga los modelos entrenados o preserve una licencia externa no registrada.  
**Error común:** confundir igualdad lógica de filas con igualdad byte a byte de dos archivos Parquet reescritos.


## 11 · Transferencia: convierte la técnica en tu propio proyecto

No memorices rutas del curso. En un proyecto nuevo identifica:

- **Archivo de entrada**: CSV o Parquet que controles y tengas permiso de usar.
- **Clave única**: ID que representa una observación.
- **Columna de agrupación**: qué necesitas comparar y por qué.
- **Etiqueta**: una variable que **exista realmente** si vas a entrenar supervisadamente.
- **Texto o variables**: de dónde provienen; ¿alguna revela directamente la etiqueta?
- **Control simple**: pandas antes de Dask.
- **Salida**: Parquet + manifiesto con fuente, conteos y decisiones.

En `Laboratorios/S13_Codespaces/analisis.py` puedes reemplazar `CSV`, `COL_TEXTO`, `COL_LABEL` y `CLASES` después de inspeccionar el nuevo esquema. Si la columna no existe, debes recibir un error visible, no fabricar valores.


## 12 · Autocomprobación técnica

| Caso | Primera alternativa |
|---|---|
| 200 filas para contar | pandas |
| Muchos archivos independientes | Dask si medir confirma beneficio |
| Un K-means pequeño | scikit-learn |
| Un solo K-means con muchos vectores | Dask-ML KMeans y medir |
| Varias configuraciones independientes | Dask Futures |
| Transformaciones Parquet en clúster Spark | PySpark (S14) |

**Distingue:** repartición de datos, ejecución de modelos independientes y entrenamiento distribuido de **un único modelo**.

## 7 · Qué modelos clásicos puede ejecutar Dask

| Modelo o trabajo | ¿Qué hace Dask? | ¿Cuándo usarlo? |
|---|---|---|
| **K-means** (`dask_ml.cluster.KMeans`) | Varios workers entrenan **un solo modelo** | Muchos vectores y operaciones suficientemente costosas |
| Regresión lineal o logística (Dask-ML) | Ajusta el modelo mediante operaciones distribuidas compatibles | Matrices grandes o complejas |
| XGBoost-Dask | Entrena un modelo supervisado entre workers | Árboles costosos y conjuntos grandes |
| Validación cruzada | Reparte **entrenamientos independientes** | Muchas configuraciones o particiones de prueba |
| DBSCAN local | No se convierte en distribuido al crear workers | Solo comparación conceptual en S13 |

**Clave:** entrenar un solo modelo distribuido es distinto de entrenar múltiples modelos. La distribución no es automáticamente más rápida. Usaremos **K-means** porque es conocido y solo requiere coordenadas numéricas.

## 8 · Datos simulados: fáciles de comprender

Genera **1.000 puntos con dos variables y cuatro grupos**. No son datos SECOP; son ejemplos sintéticos para poder ver qué significa *agrupar*.

Después puedes comparar con **20.000 registros y 20 variables** cambiando únicamente dos controles. Una matriz de esa forma `float64` ocupa **3,2 MB**, por lo que no constituye Big Data por volumen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

REGISTROS = 1000    # Para ampliar voluntariamente: 20000
VARIABLES = 2       # Para ampliar voluntariamente: 20
GRUPOS = 4
WORKERS_MODELO = 2
assert REGISTROS in (1000, 20000) and VARIABLES in (2, 20)
X_modelo, _ = make_blobs(n_samples=REGISTROS, n_features=VARIABLES,
                         centers=GRUPOS, cluster_std=1.0, random_state=42)
print("Matriz simulada:", X_modelo.shape, "| MB de valores:", round(X_modelo.nbytes/1e6, 3))
display(pd.DataFrame(X_modelo[:, :2], columns=["Variable 1", "Variable 2"]).head())
if VARIABLES == 2:
    plt.figure(figsize=(6, 3.5))
    plt.scatter(X_modelo[:, 0], X_modelo[:, 1], s=7, alpha=0.6)
    plt.title("Puntos numéricos simulados (antes de agrupar)")
    plt.xlabel("Variable 1"); plt.ylabel("Variable 2")
    plt.show()

## 9 · K-means local frente a K-means realmente distribuido

**Qué hace:** K-means asigna puntos al centroide más cercano, recalcula los centros a partir de las sumas y cantidades de cada grupo y repite. Dask-ML permite repartir ese entrenamiento entre workers.

Usamos **exactamente los mismos puntos, centroides iniciales y máximo de iteraciones**. Registramos `fit` local, arranque del clúster, transferencia de datos, `fit` Dask y predicción.

**Los dos workers son procesos de una sola máquina virtual**, no dos servidores. Si Dask resulta más lento, el resultado sigue siendo válido: debemos explicar la sobrecarga. Los nombres numéricos de los grupos pueden cambiar entre métodos.

In [ ]:
import time
import dask.array as da
from distributed import Client, LocalCluster, wait, get_task_stream
from dask_ml.cluster import KMeans as KMeansDask
from sklearn.cluster import KMeans as KMeansLocal
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits

centros_iniciales = X_modelo[np.random.default_rng(42).choice(REGISTROS, GRUPOS, replace=False)].copy()
t0 = time.perf_counter()
with threadpool_limits(limits=1):
    modelo_local = KMeansLocal(n_clusters=GRUPOS, init=centros_iniciales.copy(),
                               n_init=1, max_iter=15, random_state=42).fit(X_modelo)
fit_local = time.perf_counter() - t0

t0 = time.perf_counter()
with LocalCluster(n_workers=WORKERS_MODELO, threads_per_worker=1,
                  processes=True, dashboard_address=None) as cluster:
    with Client(cluster) as client:
        client.wait_for_workers(WORKERS_MODELO)
        trabajadores = sorted(client.scheduler_info()["workers"])
        arranque = time.perf_counter() - t0
        datos_dask = da.from_array(X_modelo, chunks=(max(100, REGISTROS//8), VARIABLES))
        t1 = time.perf_counter()
        datos_dask = client.persist(datos_dask)
        wait(datos_dask)
        persistir = time.perf_counter() - t1
        t1 = time.perf_counter()
        with get_task_stream(client=client) as tareas:
            modelo_distribuido = KMeansDask(
                n_clusters=GRUPOS, init=centros_iniciales.copy(),
                n_init=1, max_iter=15, random_state=42
            ).fit(datos_dask)
        fit_dask = time.perf_counter() - t1
        t1 = time.perf_counter()
        etiquetas_dask = np.asarray(modelo_distribuido.predict(datos_dask).compute())
        predecir_dask = time.perf_counter() - t1
        tareas_worker = {}
        for tarea in tareas.data:
            direccion = tarea.get("worker")
            if direccion:
                tareas_worker[direccion] = tareas_worker.get(direccion, 0) + 1

acuerdo = adjusted_rand_score(modelo_local.labels_, etiquetas_dask)
display(pd.DataFrame([
    {"Método":"K-means local", "Entrenar (s)":round(fit_local,3),
     "Arranque (s)":0, "Persistir (s)":0, "Inercia":round(float(modelo_local.inertia_),2)},
    {"Método":"K-means Dask", "Entrenar (s)":round(fit_dask,3),
     "Arranque (s)":round(arranque,3), "Persistir (s)":round(persistir,3),
     "Inercia":round(float(modelo_distribuido.inertia_),2)}
]))
print("Workers:", trabajadores)
print("Tareas de ajuste por worker:", tareas_worker)
print("Acuerdo entre agrupaciones (ARI):", round(acuerdo,3))
print("Dask: arranque + persistir + entrenamiento + predicción:",
      round(arranque + persistir + fit_dask + predecir_dask,3), "s")

**Cómo se lee:** `tareas_worker` identifica qué workers participaron en el ajuste; la inercia resume distancias dentro de los grupos. ARI mide acuerdo de agrupaciones, no si un grupo es útil en un negocio.

**Qué nos dice:** un modelo puede entrenarse entre workers. El tiempo de entrenamiento *solo* no equivale al costo total.

**Qué NO permite concluir:** 20.000 × 20 es un conjunto grande por número de filas, pero **pequeño por memoria**; no hay aceleración garantizada. Los puntos no son SECOP.

**Error común:** confundir dos workers con dos máquinas, o comparar un entrenamiento local contra uno distribuido ignorando el arranque.

In [ ]:
# Mostrar los grupos aprendidos, únicamente cuando hay dos variables.
if VARIABLES == 2:
    plt.figure(figsize=(6,3.5))
    plt.scatter(X_modelo[:,0], X_modelo[:,1], c=etiquetas_dask, s=8, alpha=0.65)
    centros = np.asarray(modelo_distribuido.cluster_centers_)
    plt.scatter(centros[:,0], centros[:,1], s=110, marker="X", color="black")
    plt.title("Grupos de K-means y centroides")
    plt.xlabel("Variable 1"); plt.ylabel("Variable 2")
    plt.show()
else:
    print("Gráfico de dos variables omitido: se usaron 20.")
print("Tamaños de grupos:", np.bincount(etiquetas_dask.astype(int), minlength=GRUPOS))

## 10 · Cuándo escala y cuándo no

| Matriz numérica | Memoria `float64` aproximada | Qué preguntar |
|---|---:|---|
| 1.000 × 2 | 0,016 MB | ¿El costo de Dask supera el cálculo? |
| 20.000 × 20 | 3,2 MB | ¿Por qué distribuir algo que cabe en memoria? |
| 300.000 × 20 | 48 MB | ¿Influyen más iteraciones y grupos que el tamaño? |
| 1.000.000 × 100 | 800 MB | ¿Hay memoria y CPU suficientes en cada worker? |

Memoria = registros × variables × 8 bytes; no incluye copias ni estructuras internas. Los límites reales dependen de CPU, memoria, número de grupos, iteraciones y comunicación.

**Regla de S13:** comenzar con herramientas locales; distribuir cuando exista un beneficio por capacidad o tiempo **medido**, no por moda.

In [ ]:
# Reto de cierre · respuesta estructurada y verificable (sin texto libre)
# Cambia SOLO ambos números según tu comparación de los modos texto / espera.
# Esta decisión no requiere métricas de clasificación.
# Los tiempos vienen de ESTE Codespace; su alcance está en el JSON.
ARQUITECTURA = None
LIMITE = None

OPCIONES_ARQUITECTURA = {
    1: "Secuencial cuando las tareas son demasiado pequeñas",
    2: "Dask 2 workers para lotes independientes con espera",
    3: "Comparar 4 workers y arranque antes de decidir",
}
OPCIONES_LIMITE = {
    1: "La aceleración caliente excluye arranque y preparación",
    2: "La espera OCR/ASR fue simulada, no un servicio real",
    3: "Workers de una VM comparten CPU y memoria",
    4: "Los tiempos dependen de los recursos del Codespace usado",
}
print("Elige UNA arquitectura y UN límite de los siguientes menús:")
print("Arquitectura:", OPCIONES_ARQUITECTURA)
print("Límite:", OPCIONES_LIMITE)
if ARQUITECTURA is None or LIMITE is None:
    print("Práctica pendiente: cambia ambos None por los índices seleccionados y vuelve a ejecutar.")
else:
    if type(ARQUITECTURA) is not int or ARQUITECTURA not in OPCIONES_ARQUITECTURA:
        raise ValueError("ARQUITECTURA debe ser exactamente 1, 2 o 3")
    if type(LIMITE) is not int or LIMITE not in OPCIONES_LIMITE:
        raise ValueError("LIMITE debe ser exactamente 1, 2, 3 o 4")
    # La demostración de espera puede desactivarse: no referenciar variables inexistentes.
    # Tampoco se permite exportar una decisión basada en resultados no ejecutados.
    medidas_s13 = {
        "texto": globals().get("resumen_texto"),
        "espera": globals().get("resumen_demo"),
    }
    if not all(
        isinstance(resultado, dict) and resultado.get("resultados_identicos") is True
        for resultado in medidas_s13.values()
    ):
        raise RuntimeError(
            "Falta evidencia real para uno de los dos modos. "
            "Ejecuta la celda de DEMO_DASK_EN_RUN_ALL=True y la celda "
            "de procesamiento de texto real, y vuelve a esta decisión."
        )
    import json
    from pathlib import Path
    evidencia = {
        "sesion": 13,
        "arquitectura_codigo": ARQUITECTURA,
        "arquitectura": OPCIONES_ARQUITECTURA[ARQUITECTURA],
        "limite_codigo": LIMITE,
        "limite": OPCIONES_LIMITE[LIMITE],
        "fuente": "mediciones reales de esta ejecución en Codespaces, con cronómetros separados",
        "modos_medidos": medidas_s13
    }
    salida_decision = RAIZ / "Laboratorios/S13_Codespaces/salidas/decision_s13.json"
    salida_decision.parent.mkdir(parents=True, exist_ok=True)
    salida_decision.write_text(json.dumps(evidencia,ensure_ascii=False,indent=2), encoding="utf-8")
    print("Evidencia exportada a:", salida_decision)
    print("Comprueba que la arquitectura seleccionada corresponda a tus tiempos, no a un resultado supuesto.")


## Cierre · Puente a S14

Resultados de S13: Parquet generado a partir de **SECOP real**, evidencia de Dask y un **K-means de puntos simulados** para comprender un único entrenamiento distribuido. No se mezclan los orígenes.

Aprendimos a distinguir worker de máquina física, trabajo distribuido de aceleración, y un único modelo distribuido de muchos entrenamientos independientes. S14 introduce PySpark sobre Parquet y permite comparar su K-means nativo.

Fuentes: [Dask](https://docs.dask.org/) · [Dask-ML KMeans](https://ml.dask.org/modules/generated/dask_ml.cluster.KMeans.html) · [scikit-learn KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).